In [ ]:
from dataclasses import dataclassfrom typing import List, Dict, Callable, Any, Optionalimport json, os, time, tracebackfrom litellm import completion

In [ ]:
# GOAL (clase)@dataclass(frozen=True)class Goal:    priority: int    name: str    description: str

In [ ]:
# ACTION + ActionRegistryclass Action:    def __init__(self,                 name: str,                 function: Callable,                 description: str,                 parameters: Dict,                 terminal: bool = False):        self.name = name        self.function = function        self.description = description        self.terminal = terminal        self.parameters = parameters    def execute(self, **args) -> Any:        """Execute the action's function"""        return self.function(**args)

In [ ]:
class ActionRegistry:    def __init__(self):        self.actions = {}    def register(self, action: Action):        self.actions[action.name] = action    def get_action(self, name: str) -> Optional[Action]:        return self.actions.get(name, None)    def get_actions(self) -> List[Action]:        """Get all registered actions"""        return list(self.actions.values())

In [ ]:
# Memoryclass Memory:    def __init__(self):        self.items = []  # Basic conversation history    def add_memory(self, memory: dict):        """Add memory to working memory"""        self.items.append(memory)    def get_memories(self, limit: int = None) -> List[Dict]:        """Get formatted conversation history for prompt"""        return self.items[:limit]

In [ ]:
# E - Environmentclass Environment:    def execute_action(self, action: Action, args: dict) -> dict:        """Execute an action and return the result."""        try:            result = action.execute(**args)            return self.format_result(result)        except Exception as e:            return {                "tool_executed": False,                "error": str(e),                "traceback": traceback.format_exc()            }    def format_result(self, result: Any) -> dict:        """Format the result with metadata."""        return {            "tool_executed": True,            "result": result,            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")        }

In [ ]:
class Prompt:    def __init__(self, messages: List[Dict], tools: Optional[List[Dict]] = None):        self.messages = messages        self.tools = tools or []    def to_dict(self):        return {"messages": self.messages, "tools": self.tools}

In [ ]:
# AGENT languageclass AgentLanguage:    def construct_prompt(self, actions: List[Action], environment: Environment,                         goals: List[Goal], memory: Memory) -> Prompt:        raise NotImplementedError()    def parse_response(self, response: str) -> dict:        raise NotImplementedError()

In [ ]:
class AgentFunctionCallingActionLanguage(AgentLanguage):    def format_goals(self, goals: List[Goal]) -> List[Dict]:        goal_text = "\n".join([f"- {g.name}: {g.description}" for g in sorted(goals, key=lambda x: x.priority)])        return [{            "role": "system",            "content": f"You are an AI agent with these goals:\n{goal_text}"        }]    def format_memory(self, memory: Memory) -> List[Dict]:        return [            {"role": item["type"], "content": item["content"]}            for item in memory.get_memories()        ]    def format_actions(self, actions: List[Action]) -> List[Dict]:        return [            {                "type": "function",                "function": {                    "name": action.name,                    "description": action.description[:1024],                    "parameters": action.parameters or {"type": "object", "properties": {}},                },            }            for action in actions        ]    def construct_prompt(self, actions: List[Action], environment: Environment,                          goals: List[Goal], memory: Memory) -> Prompt:        messages = []        messages.extend(self.format_goals(goals))        messages.extend(self.format_memory(memory))        tools = self.format_actions(actions)        return Prompt(messages=messages, tools=tools)    def parse_response(self, response: str) -> dict:        try:            return json.loads(response)        except Exception as e:            print(f"Failed to parse response: {str(e)}")            return {"tool": "terminate", "args": {"message": response}}

In [ ]:
# Toma todas las piezas para despues usarlas en otros metodosclass Agent:    def __init__(self,                 goals: List[Goal],                 agent_language: AgentLanguage,                 action_registry: ActionRegistry,                 generate_response: Callable[[Prompt], str],                 environment: Environment):        self.goals = goals        self.generate_response = generate_response        self.agent_language = agent_language        self.actions = action_registry        self.environment = environment    def construct_prompt(self, goals: List[Goal], memory: Memory, actions: ActionRegistry) -> Prompt:        """Build prompt with memory context"""        return self.agent_language.construct_prompt(            actions=actions.get_actions(),            environment=self.environment,            goals=goals,            memory=memory        )    def get_action(self, response):        invocation = self.agent_language.parse_response(response)        action = self.actions.get_action(invocation["tool"])        return action, invocation    def should_terminate(self, response: str) -> bool:        action_def, _ = self.get_action(response)        return action_def.terminal    def set_current_task(self, memory: Memory, task: str):        memory.add_memory({"type": "user", "content": task})    def update_memory(self, memory: Memory, response: str, result: dict):        new_memories = [            {"type": "assistant", "content": response},            {"type": "user", "content": json.dumps(result)}        ]        for m in new_memories:            memory.add_memory(m)    def prompt_llm_for_action(self, full_prompt: Prompt) -> str:        response = self.generate_response(full_prompt)        return response    def run(self, user_input: str, memory=None, max_iterations: int = 50) -> Memory:        memory = memory or Memory()        self.set_current_task(memory, user_input)        for _ in range(max_iterations):            prompt = self.construct_prompt(self.goals, memory, self.actions)            response = self.prompt_llm_for_action(prompt)            action, invocation = self.get_action(response)            result = self.environment.execute_action(action, invocation["args"])            self.update_memory(memory, response, result)            if self.should_terminate(response):                break        return memory

In [ ]:
def generate_response(prompt: Prompt) -> str:    """Call LLM to get response"""    response = completion(        model="groq/openai/gpt-oss-120b",        messages=prompt.messages,        tools=prompt.tools,        tool_choice="auto" if prompt.tools else "none",        max_tokens=1024,        reasoning_effort="low"    )    message = response.choices[0].message    if getattr(message, "tool_calls", None):        tool_call = message.tool_calls[0]        return json.dumps({            "tool": tool_call.function.name,            "args": json.loads(tool_call.function.arguments)        })    return json.dumps({"tool": "terminate", "args": {"message": message.content or ""}})

In [ ]:
# Define the agent's goalsgoals = [    Goal(priority=1, name="Gather Information", description="Read each file in the project"),    Goal(priority=1, name="Terminate", description="Call the terminate call when you have read all the files "                                                    "and provide the content of the README in the terminate message")]

In [ ]:
def read_project_file(name: str) -> str:    with open(name, "r") as f:        return f.read()def list_project_files() -> List[str]:    return sorted([file for file in os.listdir(".") if file.endswith(".py")])# Define the action registry and register some actionsaction_registry = ActionRegistry()action_registry.register(Action(    name="list_project_files",    function=list_project_files,    description="Lists all files in the project.",    parameters={},    terminal=False))action_registry.register(Action(    name="read_project_file",    function=read_project_file,    description="Reads a file from the project.",    parameters={        "type": "object",        "properties": {            "name": {"type": "string"}        },        "required": ["name"]    },    terminal=False))action_registry.register(Action(    name="terminate",    function=lambda message: f"{message}\nTerminating...",    description="Terminates the session and prints the message to the user.",    parameters={        "type": "object",        "properties": {            "message": {"type": "string"}        },        "required": []    },    terminal=True))

In [ ]:
agent_language = AgentFunctionCallingActionLanguage()

In [ ]:
# Define the environmentenvironment = Environment()# Create an agent instanceagent = Agent(goals, agent_language, action_registry, generate_response, environment)# Run the agent with user inputuser_input = "Write a README for this project."final_memory = agent.run(user_input)# Print the final memoryprint(final_memory.get_memories())